# Introduction to AI-Assisted Coding: Prompt Engineering with LLMs in Google Colab

## Overview
This hands-on workshop introduces participants to using large language models (LLMs) as interactive coding assistants. Using Gemini within Google Colab, participants will learn how to guide AI systems through prompt engineering to generate, modify, and debug Python code while developing analytical workflows. The workshop focuses on practical strategies for interacting with LLMs as coding partners without requiring advanced programming expertise.

Geospatial data analysis will be used as a case study to demonstrate these techniques. Participants will work through an example that integrates satellite remote sensing data from NASA’s Harmonized Landsat–Sentinel (HLS) NDVI product with U.S. Census socioeconomic datasets to explore environmental justice indicators. Through this example, participants will develop a simple workflow in Google Colab that demonstrates how LLM-assisted coding can support data integration, analysis, and visualization.

By the end of the session, participants will have gained practical experience collaborating with LLMs as coding assistants and will understand how similar approaches can be applied to their own data analysis workflows.


## Learning Objectives
- Use large language models (LLMs) to generate and modify Python code in Google Colab
- Apply prompt-engineering strategies to iteratively debug code, resolve errors, and improve analytical workflows
- Integrate satellite remote sensing data (NDVI from NASA’s Harmonized Landsat–Sentinel dataset) with U.S. Census socioeconomic datasets
- Summarize and visualize spatial patterns using maps, tables, and simple statistical comparisons
- Develop skills for using AI to write and refine code while creating a reproducible workflow that can be adapted to other geospatial datasets and research questions.

## Prerequisites
- Basic familiarity with data analysis or geospatial concepts (no coding experience required)
- A laptop with internet access
- A Google account to access Google Colab
- A great attitude!

## Part 1: Brief Introduction to Colab and Setting Up Your Environment and Working Directory

### Key Tasks
- Introduction to Google Colab and its features
- Install a suite of packages we will use in our workspace
- Set up our working directory
---



In [ ]:
# (Optional) To mount your Google Drive if you want to work from a Google Drive folder etc
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# 1. Clone the repo to the fast, temporary storage
!git clone https://github.com/CGC-UMCES/NAIRR-2026-Tutorial-Intro-Colab-LLM-Coding.git

In [ ]:
# 2. Change directory into the new repo folder
%cd /content/NAIRR-2026-Tutorial-Intro-Colab-LLM-Coding/

# 3. (Optional) Check to see your files are there
!ls

In [ ]:
# (Optional) Install required packages
# Not important when in Colab but important for sharing your notebook externally!!
# !pip install pandas geopandas numpy matplotlib seaborn scikit-learn rasterio earthpy rasterstats

####<font color="#FF69B4">PROMPT: Check the working directory for my notebook and print the contents</font>





In [ ]:
import os

# Check the current working directory
current_directory = os.getcwd()
print(f"Current working directory: {current_directory}")

# List the contents of the current working directory
print("\nContents of the current working directory:")
for item in os.listdir(current_directory):
    print(item)

## Part 2: Working with Census Data to Create a Social Vulnerability Index (SVI)

In this section, you'll work with a subset of the 2022 Census data from the Baltimore, MD region and create a basic social vulnerability index.

### Key Tasks
- Read in a .geojson file (essentially a spatial dataframe like a shapefile)
- Learn to investigate variables
- Create and view a simple social vulnerability index (SVI)


---



####<font color="#FF69B4">PROMPT: Import a .geojson file with census data in it named baltimore_region_acs_2022_epsg4269.geojson from my working directory and get some info about it. I would like the variable to be called gdf</font>

In [ ]:
import geopandas as gpd

# Import .geojson file as census_df
census_loc = '/content/NAIRR-2026-Tutorial-Intro-Colab-LLM-Coding/baltimore_region_acs_2022_epsg4269.geojson'
gdf = gpd.read_file(census_loc)

# Display the first few rows to verify
print(gdf.head())

# Look at the general info of the file (columns names, types, etc)
gdf.info()


####<font color="#FF69B4">PROMPT: Calculate three more metrics that include the proportion of people in poverty, the proportion of renters, and the proportion of the population that is nonwhite and add to gdf<font>

Specifically calulate them in the following way:

gdf['prop_poverty'] = (gdf['population_below_poverty'] / gdf['total_population'])

gdf['prop_renter'] = (gdf['renter_occupied_units'] / gdf['total_occupied_units'])

gdf['prop_nonwhite'] = 1 - (gdf['white_population'] / gdf['total_population'])
<font>

In [ ]:
# Calculate proportion metrics for poverty, renter, and nonwhite populations
gdf['prop_poverty'] = (gdf['population_below_poverty'] / gdf['total_population'])
gdf['prop_renter'] = (gdf['renter_occupied_units'] / gdf['total_occupied_units'])
gdf['prop_nonwhite'] = 1 - (gdf['white_population'] / gdf['total_population'])

####<font color="skyblue">CHALLENGE: Can you prompt the LLM to show only the first 10 rows of gdf and only for the new columns you just created?</font>

In [ ]:
# Select the first 10 rows and only the new columns
print(gdf[['prop_poverty', 'prop_renter', 'prop_nonwhite']].head(10))

####<font color="#FF69B4">PROMPT: Plot the data from the three new columns we just created side-by-side using the viridis colormap<font>

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with 3 subplots side-by-side
fig, axes = plt.subplots(1, 3, figsize=(20, 10))

# Plot Poverty Rate
gdf.plot(column='prop_poverty',
         cmap='viridis',
         legend=True,
         legend_kwds={'label': 'Proportion Below Poverty'},
         ax=axes[0])
axes[0].set_title('Proportion Below Poverty by Census Tract')
axes[0].set_axis_off() # Turn off axes for a cleaner map

# Plot Renter Proportion
gdf.plot(column='prop_renter',
         cmap='viridis',
         legend=True,
         legend_kwds={'label': 'Proportion Renter Occupied'},
         ax=axes[1])
axes[1].set_title('Proportion Renter Occupied by Census Tract')
axes[1].set_axis_off() # Turn off axes for a cleaner map

# Plot Proportion Nonwhite
gdf.plot(column='prop_nonwhite',
         cmap='viridis',
         legend=True,
         legend_kwds={'label': 'Proportion Nonwhite'},
         ax=axes[2])
axes[2].set_title('Proportion Nonwhite by Census Tract')
axes[2].set_axis_off() # Turn off axes for a cleaner map

# Adjust layout and display the plot
plt.tight_layout()
plt.show()

####<font color="skyblue">CHALLENGE: Is there anything you would change about the way the plots look (e.g., colorbar size, font size, colormap, etc)? Ask the LLM and customize away! NOTE: If you get errors you can hit the "Explain Error" button in the lower left after the error and feed it right back into Gemini who will then try to correct your code!</font>

####<font color="#FF69B4">PROMPT: Change the colormaps to "turbo", reduce the size of the colorbars, and make the font slightly larger<font>

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with 3 subplots side-by-side
fig, axes = plt.subplots(1, 3, figsize=(20, 10))

# Define common legend properties (removed 'size')
legend_properties = {} # Adjust size as needed

# Plot Poverty Rate
# Store the Axes object returned by plot
ax1 = gdf.plot(column='prop_poverty',
         cmap='turbo',  # Changed colormap
         legend=True,
         legend_kwds={'label': 'Proportion Below Poverty', **legend_properties, 'shrink': 0.5}, # Adjusted legend properties and shrink
         ax=axes[0])
axes[0].set_title('Proportion Below Poverty by Census Tract', fontsize=14) # Adjusted title font size
axes[0].set_axis_off() # Turn off axes for a cleaner map

# Plot Renter Proportion
# Store the Axes object returned by plot
ax2 = gdf.plot(column='prop_renter',
         cmap='turbo',  # Changed colormap
         legend=True,
         legend_kwds={'label': 'Proportion Renter Occupied', **legend_properties, 'shrink': 0.5}, # Adjusted legend properties and shrink
         ax=axes[1])
axes[1].set_title('Proportion Renter Occupied by Census Tract', fontsize=14) # Adjusted title font size
axes[1].set_axis_off() # Turn off axes for a cleaner map

# Plot Proportion Nonwhite
# Store the Axes object returned by plot
ax3 = gdf.plot(column='prop_nonwhite',
         cmap='turbo',  # Changed colormap
         legend=True,
         legend_kwds={'label': 'Proportion Nonwhite', **legend_properties, 'shrink': 0.5}, # Adjusted legend properties and shrink
         ax=axes[2])
axes[2].set_title('Proportion Nonwhite by Census Tract', fontsize=14) # Adjusted title font size
axes[2].set_axis_off() # Turn off axes for a cleaner map

# If you need to adjust colorbar font size after plotting:
# This requires accessing the colorbar object, which is not directly returned by gdf.plot.
# A more complex approach involving creating an explicit colorbar axis might be needed
# if fine-grained control over colorbar font size is essential with gdf.plot legend=True.
# For simplicity, removing the 'size' keyword is the direct fix for the Type Error.


# Adjust layout and display the plot
plt.tight_layout()
plt.show()

####<font color="#FF69B4">PROMPT: Create a Social Vulnerability Index (SVI) from my three new metrics and plot it</font>

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

# Create a Social Vulnerability Index (SVI)
# Normalize and combine poverty, income, and housing metrics
# Select variables for SVI
svi_vars = ['prop_poverty', 'prop_renter', 'prop_nonwhite'] # Use the proportion columns
svi_df = gdf[svi_vars].copy()

# Handle missing values
svi_df.fillna(svi_df.mean(), inplace=True)

# Normalize the variables (0-1 scale)
scaler = MinMaxScaler()
svi_normalized = scaler.fit_transform(svi_df)
svi_normalized_df = pd.DataFrame(svi_normalized, columns=svi_vars, index=svi_df.index)

# Calculate SVI (simple average of normalized variables)
gdf['social_vulnerability_index'] = svi_normalized_df.mean(axis=1)

# Visualize social vulnerability
fig, ax = plt.subplots(1, figsize=(15, 10))
gdf.plot(column='social_vulnerability_index',
                cmap='viridis',
                legend=True,
                legend_kwds={'label': 'Social Vulnerability Index'},
                ax=ax)
plt.title('Social Vulnerability by Census Tract')
plt.savefig('social_vulnerability_map.png', dpi=300, bbox_inches='tight')
plt.show()

####<font color="#FF69B4">PROMPT: What did the MinMaxScaler do to the data? Can you compare the values from before and after it was applied?</font>

In [ ]:
# Display the head of the data BEFORE MinMaxScaler
print("Data BEFORE MinMaxScaler:")
print(svi_df.head())

print("\n" + "="*30 + "\n") # Separator for clarity

# Display the head of the data AFTER MinMaxScaler
print("Data AFTER MinMaxScaler:")
print(svi_normalized_df.head())

####<font color="#FF69B4">PROMPT: Plot a histogram of the SVI data</font>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd # Import pandas in case it wasn't imported earlier in this block

# Create a histogram of the social_vulnerability_index column
plt.figure(figsize=(10, 6)) # Set the size of the figure
sns.histplot(gdf['social_vulnerability_index'].dropna(), # Select the column and drop any potential missing values
             kde=True, # Add a Kernel Density Estimate line for a smoother distribution shape
             bins=50) # Increase the number of bins for more detail

plt.title('Distribution of Social Vulnerability Index') # Set the title of the plot
plt.xlabel('Social Vulnerability Index') # Set the x-axis label
plt.ylabel('Frequency') # Set the y-axis label
plt.grid(axis='y', alpha=0.75) # Add a grid for easier reading
plt.show() # Display the plot

## Part 3: Using Detailed Prompts to Integrate Harmonized Landsat Sentinel-2 NDVI Data with Census Data to Illuminate Patterns of Social Vulnerability

In this section, you will work with a summer 2022 Normalized Difference Vegetation Index (NDVI) composite dataset derived from Harmonized Landsat Sentinel-2 [(HLS info)](https://hls.gsfc.nasa.gov/) and integrate it with the 2022 Census data for deeper exploration in the Baltimore, MD region.

### Key Tasks
- Import and view a geotiff raster file
- Extract NDVI statistics for each census tract and merge data into new colums
- Calculate a Green Space Inequity Index and view it
- Create a list of high priority environmental justice areas

### NDVI Primer
NDVI (Normalized Difference Vegetation Index) is a widely used remote sensing index that quantifies vegetation health and density using satellite or aerial imagery. It is calculated from the visible and near-infrared light reflected by vegetation.
<br>
<br>

📊 NDVI Formula:
$$
\text{NDVI} = \frac{(NIR - RED)}{(NIR + RED)}
$$

- NIR = Reflectance in the near-infrared spectrum (plants strongly reflect this light)

- RED = Reflectance in the red spectrum (plants absorb this light for photosynthesis)
<br>

🌱 NDVI Values:
NDVI values range from -1 to +1, and they indicate:

| NDVI Value    | Interpretation                           |
| ------------- | ---------------------------------------- |
| **< 0**       | Water, clouds, snow, or bare soil        |
| **0.1 – 0.2** | Sparse vegetation (e.g., shrubs)         |
| **0.2 – 0.5** | Moderate vegetation (e.g., grassland)    |
| **0.5 – 0.9** | Dense, healthy vegetation (e.g., forest) |
<br>
<br>

🛰️ Why is NDVI Useful?
- Environmental Monitoring: Assess drought, deforestation, and land degradation.
- Agriculture: Monitor crop health and stress.
- Urban Planning: Evaluate green space distribution.
- Climate Studies: Understand carbon sinks and land surface processes.

---

####<font color="#FF69B4">PROMPT: Load my NDVI geotiff from the working directory and plot the data</font>

In [ ]:
import rasterio
import matplotlib.pyplot as plt

# Path to the NDVI geotiff
ndvi_path = '/content/NAIRR-2026-Tutorial-Intro-Colab-LLM-Coding/baltimore_region_HLSS30_mean_summer_2022_NDVI_epsg4269_clip.tif'

# Open the GeoTIFF
try:
    with rasterio.open(ndvi_path) as src:
        ndvi = src.read(1)  # Read the first band
        ndvi_transform = src.transform # Get the affine transform
        ndvi_crs = src.crs # Get the CRS

    print("GeoTIFF opened successfully.")
    print(f"GeoTIFF CRS: {ndvi_crs}")
    # You might want to compare this to your gdf CRS:
    # print(f"GeoDataFrame CRS: {gdf.crs}") # Assuming gdf is already loaded

    # Plot the NDVI data
    fig, ax = plt.subplots(1, 1, figsize=(15, 10))
    im = ax.imshow(ndvi, cmap='YlGn', vmin=-1, vmax=1) # Set value range from -1 to 1
    fig.colorbar(im, ax=ax, label='NDVI')
    plt.title('Mean Summer 2022 NDVI - Baltimore Region')
    plt.xlabel('Column #') # Pixel column index
    plt.ylabel('Row #') # Pixel row index
    plt.show()

except rasterio.errors.RasterioIOError as e:
    print(f"Error opening GeoTIFF: {e}")
    print("Please check the file path and ensure the file exists.")
except Exception as e:
    print(f"An error occurred during plotting: {e}")

####<font color="#FF69B4">PROMPT: Extract statistics from the ndvi dataset for each tract in my census data and add the data as new columns into my census dataset gdf. Also make sure that the nodata is set for all values equal to -9999 so they aren't being considered</font>

In [ ]:
# Extract zonal NDVI statistics for each census tract in gdf

# Import rasterstats
import rasterstats
import pandas as pd # Import pandas if not already available in this block

# Stats to calculate
stats_to_calculate = ['mean', 'min', 'max', 'median', 'count'] # Example stats

# Extract zonal statistics
zonal_statistics = rasterstats.zonal_stats(
    gdf.geometry,       # Geometries from the GeoDataFrame
    ndvi,       # Raster data (NumPy array)
    affine=ndvi_transform, # Affine transform of the raster
    stats=stats_to_calculate, # List of statistics to calculate
    nodata=-9999 # Specify the no-data value
    )

print(f"Calculated zonal statistics for {len(zonal_statistics)} features.")

# Convert the list of dictionaries to a pandas DataFrame
stats_df = pd.DataFrame(zonal_statistics)

# Append the new columns to the original GeoDataFrame
# Assuming the order of results matches the order of geometries in gdf
gdf = gdf.join(stats_df)

# Rename the new columns for clarity (optional but recommended)
new_col_names = {stat: f'ndvi_{stat}' for stat in stats_to_calculate}
gdf = gdf.rename(columns=new_col_names)

# Display the first few rows to see the new columns
print(gdf.head())

####<font color="#FF69B4">PROMPT: Create a new green space inequity index that uses the SVI variables and the ndvi data and plot the data on a scatter plot</font>

In [ ]:
# Calculate Green Space Inequity Index
# Higher index means less green space relative to social vulnerability
gdf['green_space_inequity'] = (1 - gdf['ndvi_mean']) * gdf['social_vulnerability_index']

# Visualize the relationship between social vulnerability and NDVI
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    gdf['social_vulnerability_index'],
    gdf['ndvi_mean'],
    c=gdf['green_space_inequity'],
    cmap='viridis',
    alpha=0.7,
    s=50
)
plt.colorbar(scatter, label='Green Space Inequity Index')
plt.xlabel('Social Vulnerability Index')
plt.ylabel('Mean NDVI (Vegetation Index)')
plt.title('Relationship Between Social Vulnerability and Vegetation Cover')
plt.tight_layout()
# plt.savefig('vulnerability_vs_ndvi.png', dpi=300)
plt.show()

# Map the Green Space Inequity Index
fig, ax = plt.subplots(1, figsize=(15, 10))
gdf.plot(
    column='green_space_inequity',
    cmap='RdYlGn_r',  # Reversed colormap (red = high inequity)
    legend=True,
    legend_kwds={'label': 'Green Space Inequity Index'},
    ax=ax
)
plt.title('Green Space Inequity by Census Tract')
plt.savefig('green_space_inequity_map.png', dpi=300, bbox_inches='tight')
plt.show()

####<font color="#FF69B4">PROMPT: Plot maps of SVI and the Green Space Inequity Index next to each other to compare</font>:

In [ ]:
import matplotlib.pyplot as plt

# Choose a colormap to use for both plots
common_cmap = 'viridis' # Or use 'RdYlGn_r' or any other colormap

# Create a figure with 2 subplots side-by-side
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Plot Social Vulnerability Index
gdf.plot(column='social_vulnerability_index',
         cmap=common_cmap, # Use the chosen common colormap
         legend=True,
         legend_kwds={'label': 'Social Vulnerability Index'},
         ax=axes[0])
axes[0].set_title('Social Vulnerability by Census Tract')
axes[0].set_axis_off()

# Plot Green Space Inequity Index
gdf.plot(column='green_space_inequity',
         cmap=common_cmap,  # Use the chosen common colormap
         legend=True,
         legend_kwds={'label': 'Green Space Inequity Index'},
         ax=axes[1])
axes[1].set_title('Green Space Inequity by Census Tract')
axes[1].set_axis_off()

# Adjust layout and display the plot
plt.tight_layout()
plt.show()

####<font color="#FF69B4">PROMPT: Plot a correlation matrix for the variables that go into the green space inequity index?</font>


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Select the columns for the correlation matrix
# These are the three proportion metrics and the mean NDVI
correlation_vars = ['prop_poverty', 'prop_renter', 'prop_nonwhite', 'ndvi_mean']

# Create a subset DataFrame with only these columns
corr_df = gdf[correlation_vars].copy()

# Calculate the correlation matrix
# .corr() method calculates the pairwise correlation of columns, excluding NA/null values
correlation_matrix = corr_df.corr()

# Print the correlation matrix
print("Correlation Matrix:")
print(correlation_matrix)

# Optional: Visualize the correlation matrix using a heatmap
plt.figure(figsize=(8, 6)) # Set figure size for the heatmap
sns.heatmap(correlation_matrix,
            annot=True,     # Annotate cells with the correlation values
            cmap='coolwarm', # Colormap (e.g., coolwarm, viridis, plasma)
            fmt=".2f",      # Format the annotations to 2 decimal places
            linewidths=.5)  # Add lines between cells for clarity

plt.title('Correlation Matrix of Proportion Metrics and Mean NDVI') # Set the title
plt.show() # Display the plot

####<font color="#FF69B4">PROMPT: Make a plot showing only the census tracts with the 10 highest environmental justice priority scores based on the green space inequity index?</font>:

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Calculate a priority score based on the green space inequity index.
# You can adjust this formula based on how you want to weight inequity.
# For simplicity, we'll just use the green_space_inequity directly for ranking.
# If you wanted to combine other factors, you would do it here.
gdf['priority_score'] = gdf['green_space_inequity']

# Handle any potential NaN values in 'priority_score' before sorting
gdf_sorted = gdf.dropna(subset=['priority_score']).sort_values(by='priority_score', ascending=False)

# Select the top 10 high-priority tracts
top_10_tracts = gdf_sorted.head(10).copy()

# Print the top 10 tracts and their scores
print("Top 10 High-Priority Census Tracts:")
print(top_10_tracts[['tract_code', 'priority_score', 'social_vulnerability_index', 'ndvi_mean']]) # Display relevant info

# Create a base map of all census tracts
fig, ax = plt.subplots(1, figsize=(15, 10))
gdf.plot(ax=ax, color='lightgray', edgecolor='white', linewidth=0.5)

# Plot the top 10 high-priority tracts on top
top_10_tracts.plot(
    column='priority_score',
    cmap='OrRd',  # Use a color map to show score differences among the top 10
    legend=True,
    legend_kwds={'label': 'Priority Score'},
    ax=ax,
    edgecolor='black' # Add an edge color to make them stand out
)

# Add titles and turn off axes
plt.title('Census Tracts with Top 10 Highest Environmental Justice Priority Scores')
ax.set_axis_off()

# Show the plot
plt.show()

## Part 4: Go Wild with LLMs!!

### Key Tasks
- Have fun exploring the data further in anyway you see fit using LLMs as your guide!
- I have provided a clustering example prompt with the code available in the cheat sheet.
- Test your new Colab, Python, and LLM skills!
<br>

### Suggestions for exploring the data with LLMs
- Ask LLM to convert parts or all of this code into R, Matlab, etc
- Run principal component analysis on the data
- Try Google Colab with some of your own data
- Try to build a script to download data from an API like the Census Data API (you will likely need register to get an API key) [Census API User Guide](https://www.census.gov/data/developers/guidance/api-user-guide.html)

####<font color="#FF69B4">PROMPT: Run a cluster analysis that includes the variables that went into the SVI along with NDVI variables and plot the results</font>

In [ ]:
# Cluster analysis to identify similar environmental justice patterns
from sklearn.cluster import KMeans
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler # Make sure MinMaxScaler is imported

# Prepare data for clustering
# Select the SVI variables ('renter_rate', 'poverty_rate', 'pct_nonwhite') and 'ndvi_median'
cluster_data = gdf[['ndvi_median', 'prop_poverty', 'prop_renter', 'prop_nonwhite']].copy() # Corrected column names to use prop_*
cluster_data = cluster_data.dropna()  # Remove rows with missing values

# Normalize the data
scaler = MinMaxScaler() # Ensure scaler is defined (it was defined earlier, but good to be explicit here)
cluster_data_scaled = scaler.fit_transform(cluster_data)
cluster_data_scaled_df = pd.DataFrame(cluster_data_scaled, columns=cluster_data.columns, index=cluster_data.index) # Convert back to DataFrame to keep index

# Determine optimal number of clusters using the elbow method
inertia = []
k_range = range(1, 11) # Check k from 1 to 10
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10) # Added n_init for KMeans robustness
    kmeans.fit(cluster_data_scaled_df)
    inertia.append(kmeans.inertia_)

# Plot elbow curve
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, 'o-')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia (Within-cluster sum of squares)')
plt.title('Elbow Method for Optimal k')
plt.grid(True)
plt.show()


In [ ]:
# --- Choose an optimal number of clusters based on the elbow curve ---
# After reviewing the elbow curve, you'll select a value for k.
# For example, if the elbow seems to be at k=4:
# k = 4 # Replace with your chosen optimal k
# Let's choose k=4 as an example to continue

k = 4 # Example: Based on a hypothetical elbow at k=4

# Run KMeans with the chosen number of clusters
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(cluster_data_scaled_df) # Fit and predict on the scaled data

# Add cluster information to the original GeoDataFrame
# We need to add the cluster assignments back to the original gdf based on the index
gdf['cluster'] = None # Initialize the column

# Make sure the indices align before joining. If not, re-index the cluster assignments.
# A safe way is to create a Series from the cluster assignments with the same index
cluster_series = pd.Series(clusters, index=cluster_data_scaled_df.index, name='cluster')

# Assign the cluster values back to the original gdf based on the index
gdf.loc[cluster_data_scaled_df.index, 'cluster'] = cluster_series

# Convert the cluster column to a categorical type for better plotting
gdf['cluster'] = gdf['cluster'].astype('category')


# Visualize the clusters on the map
fig, ax = plt.subplots(1, figsize=(15, 10))
# Ensure the 'cluster' column is treated as categorical
gdf.plot(
    column='cluster',
    cmap='tab10', # tab10 is a good categorical colormap
    categorical=True,
    legend=True,
    ax=ax
)
plt.title('Environmental Justice Pattern Clusters')
ax.set_axis_off()
plt.show()


In [ ]:
# Analyze cluster characteristics using the SCALED data
# Calculate the mean of the SCALED variables within each cluster

# Ensure 'cluster' column exists in the scaled DataFrame.
# Since we added 'cluster' to the original gdf based on the scaled_df index,
# we can join the cluster column back to the scaled df before grouping.
cluster_data_scaled_df_with_clusters = cluster_data_scaled_df.copy()

# Join the cluster assignments to the scaled DataFrame (using the previously created series)
cluster_data_scaled_df_with_clusters['cluster'] = cluster_series

# Now group by the cluster column
cluster_profiles_scaled = cluster_data_scaled_df_with_clusters.groupby('cluster')[
    ['ndvi_median', 'prop_poverty', 'prop_renter', 'prop_nonwhite'] # Corrected column names
].mean()

print("\nCluster Profiles (Mean SCALED values for each variable by cluster):")
print(cluster_profiles_scaled)

# Visualize cluster profiles (using scaled means)
cluster_profiles_scaled.plot(
    kind='bar',
    figsize=(12, 8),
    colormap='viridis' # Or another suitable colormap
)
plt.title('Average Characteristics of Environmental Justice Clusters (Scaled Data)')
plt.ylabel('Average Scaled Value')
plt.xticks(rotation=45, ha='right') # Rotate labels for readability
plt.tight_layout()
plt.show()

####<font color="#FF69B4">PROMPT: Help me interpret these cluster profiles in a descriptive way?

HINT: Try copying the table values printed out above into Gemini and asking to interpret the data</font>


####<font color="#FF69B4">PROMPT: Plot cluster vs income?</font>




In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd # Ensure pandas is imported

# Make sure the 'cluster' column is created and populated in gdf
# This should have happened in the previous clustering steps.
# If not, you would need to re-run those steps.

# Ensure 'median_household_income' column exists and is numeric
if 'median_household_income' not in gdf.columns:
    print("Error: 'median_household_income' column not found in gdf. Please check your data.")
elif not pd.api.types.is_numeric_dtype(gdf['median_household_income']):
     print("Error: 'median_household_income' column is not numeric. Please check data types.")
else:
    # Create a box plot to show the distribution of median household income for each cluster
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='cluster', y='median_household_income', data=gdf.dropna(subset=['cluster', 'median_household_income'])) # Drop rows with missing cluster or income

    plt.title('Median Household Income Distribution by Environmental Justice Cluster')
    plt.xlabel('Cluster')
    plt.ylabel('Median Household Income')
    plt.grid(axis='y', alpha=0.75)
    plt.show()

    # Optional: Calculate and print the mean median household income for each cluster
    mean_household_income_by_cluster = gdf.groupby('cluster')['median_household_income'].mean()
    print("\nMean Median Household Income by Cluster:")
    print(mean_household_income_by_cluster)


####<font color="#FF69B4">PROMPT: Plot NDVI vs income with points colored by the Green Space Inequity Index</font>


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd # Ensure pandas is imported

# Ensure the required columns exist and are numeric
required_cols = ['median_household_income', 'ndvi_mean', 'green_space_inequity']
if not all(col in gdf.columns for col in required_cols):
    print(f"Error: One or more required columns {required_cols} not found in gdf. Please check your data.")
elif not all(pd.api.types.is_numeric_dtype(gdf[col]) for col in required_cols):
    print("Error: One or more of the required columns are not numeric. Please check data types.")
else:
    # Create the scatter plot colored by Green Space Inequity
    plt.figure(figsize=(10, 6))

    scatter = plt.scatter(
        gdf['median_household_income'],  # X-axis: Income
        gdf['ndvi_mean'],              # Y-axis: Mean NDVI
        c=gdf['green_space_inequity'],  # Color: Green Space Inequity Index
        cmap='RdYlGn_r',               # Colormap (Red=High Inequity, Green=Low Inequity)
        alpha=0.7,                     # Adjust transparency
        s=50                           # Adjust point size
    )

    # Add a color bar
    cbar = plt.colorbar(scatter)
    cbar.set_label('Green Space Inequity Index')

    plt.title('Mean NDVI vs. Median Household Income, Colored by Green Space Inequity')
    plt.xlabel('Median Household Income')
    plt.ylabel('Mean NDVI (Vegetation Index)')
    plt.grid(True, linestyle='--', alpha=0.6)

    # Optional: Add annotations for specific points if needed

    plt.show()

## Part 5: Creating a Shareable Interactive Data Dashboard

In this final section, I'll show you how to create an  interactive data dashboard within Google Colab that can be shared live with colleagues and collaborators.




#### <font color="#FF69B4">PROMPT: Build an interactive data dashboard with the following structure:
1) The left panel should be a map of the Census blocks colored by income values
2) The right panel should be a scatter plot with mean NDVI on the Y-axis and SVI on the X-axis with the points colored by the Green Space Inequity Index
3) The scatter plot points should be able to be selected using a lasso tool with those points highlighted on the map on the left panel</font>

In [ ]:
# @title 🌲 Green Space Inequity & Income Dashboard {display-mode: "form"}

import json
import pandas as pd
from IPython.display import HTML
from google.colab import output

# 1. Prepare Data
dashboard_gdf = gdf.copy()
dashboard_gdf['geom_json'] = dashboard_gdf.geometry.apply(lambda x: x.__geo_interface__)

data_cols = ['tract_code', 'median_household_income', 'social_vulnerability_index', 'ndvi_mean', 'green_space_inequity', 'geom_json']
json_data = dashboard_gdf[data_cols].to_json(orient='records')

def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

html_content = """
<!DOCTYPE html>
<html>
<head>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <script src="https://cdn.plot.ly/plotly-2.24.1.min.js"></script>
    <style>
        body { font-family: sans-serif; margin: 0; background: #f4f6f8; color: #333; text-align: left; }
        .header { background: #fff; padding: 15px 25px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); text-align: left; }
        .container { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; padding: 20px; height: 650px; }
        .card { background: #fff; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); overflow: hidden; display: flex; flex-direction: column; text-align: left; }
        .card-header { padding: 12px 15px; border-bottom: 1px solid #eee; font-weight: bold; background: #fafafa; text-align: left; }
        #map { flex-grow: 1; }
        .canvas-wrapper { position: relative; flex-grow: 1; min-height: 0; }
        #scatter { height: 100%; width: 100%; }
        .info.legend { background: white; padding: 10px; line-height: 18px; color: #555; border-radius: 5px; box-shadow: 0 0 15px rgba(0,0,0,0.2); text-align: left; }
        .info.legend i { width: 18px; height: 18px; float: left; margin-right: 8px; opacity: 0.7; }
        .info.legend strong { display: block; margin-bottom: 5px; text-align: left; }
    </style>
</head>
<body>
    <div class="header">
        <h2 style="margin:0;">Interactive Green Space Inequity Analysis</h2>
        <p style="margin:5px 0 0 0; color: #666;">Lasso points on the scatter plot to filter/highlight Census Tracts on the map.</p>
    </div>
    <div class="container">
        <div class="card">
            <div class="card-header">Map: Median Household Income</div>
            <div id="map"></div>
        </div>
        <div class="card">
            <div class="card-header">Scatter: NDVI vs. SVI (Colored by Inequity)</div>
            <div class="canvas-wrapper">
                <div id="scatter"></div>
            </div>
        </div>
    </div>

    <script>
        window.onerror = function(message) {
            google.colab.kernel.invokeFunction('report_js_error', [message], {});
        };

        const rawData = DATA_PLACEHOLDER;

        function getIncomeColor(d) {
            return d > 120000 ? '#006837' :
                   d > 90000  ? '#31a354' :
                   d > 70000  ? '#78c679' :
                   d > 50000  ? '#c2e699' :
                                '#ffffcc';
        }

        const map = L.map('map').setView([39.2904, -76.6122], 10);
        L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png').addTo(map);

        let geoJsonLayer;
        const tractLayers = {};

        function initMap() {
            geoJsonLayer = L.geoJson(rawData.map(d => ({
                type: "Feature",
                id: d.tract_code,
                geometry: d.geom_json,
                properties: d
            })), {
                style: (feature) => ({
                    fillColor: getIncomeColor(feature.properties.median_household_income),
                    weight: 1,
                    opacity: 1,
                    color: 'white',
                    fillOpacity: 0.7
                }),
                onEachFeature: (feature, layer) => {
                    tractLayers[feature.id] = layer;
                    layer.bindPopup(`Tract: ${feature.id}<br>Income: $${feature.properties.median_household_income.toLocaleString()}`);
                }
            }).addTo(map);

            const legend = L.control({position: 'bottomleft'});
            legend.onAdd = function (map) {
                const div = L.DomUtil.create('div', 'info legend'),
                    grades = [0, 50000, 70000, 90000, 120000];
                div.innerHTML += '<strong>Income ($)</strong>';
                for (let i = 0; i < grades.length; i++) {
                    div.innerHTML +=
                        '<i style="background:' + getIncomeColor(grades[i] + 1) + '"></i> ' +
                        grades[i].toLocaleString() + (grades[i + 1] ? '&ndash;' + grades[i + 1].toLocaleString() + '<br>' : '+');
                }
                return div;
            };
            legend.addTo(map);
        }

        function initScatter() {
            const trace = {
                x: rawData.map(d => d.social_vulnerability_index),
                y: rawData.map(d => d.ndvi_mean),
                text: rawData.map(d => d.tract_code),
                mode: 'markers',
                type: 'scatter',
                marker: {
                    size: 8,
                    color: rawData.map(d => d.green_space_inequity),
                    colorscale: 'Viridis',
                    showscale: true,
                    colorbar: { title: 'Inequity Index', thickness: 15, xanchor: 'left', titleside: 'top' }
                }
            };
            const layout = {
                margin: { t: 10, r: 10, b: 40, l: 50 },
                xaxis: { title: 'Social Vulnerability Index (SVI)', titlefont: { size: 12 }, tickfont: { size: 10 } },
                yaxis: { title: 'Mean NDVI', titlefont: { size: 12 }, tickfont: { size: 10 } },
                dragmode: 'lasso',
                hovermode: 'closest',
                legend: { x: 0, y: 1, xanchor: 'left' }
            };
            Plotly.newPlot('scatter', [trace], layout, {responsive: true});

            document.getElementById('scatter').on('plotly_selected', (eventData) => {
                geoJsonLayer.eachLayer(layer => {
                    layer.setStyle({ fillOpacity: 0.1, weight: 0.5 });
                });
                if (eventData && eventData.points.length > 0) {
                    eventData.points.forEach(pt => {
                        const id = pt.text;
                        if (tractLayers[id]) {
                            tractLayers[id].setStyle({ fillOpacity: 0.9, weight: 2, color: '#000' });
                        }
                    });
                } else {
                    geoJsonLayer.eachLayer(layer => {
                        layer.setStyle({ fillOpacity: 0.7, weight: 1, color: 'white' });
                    });
                }
            });
        }

        initMap();
        initScatter();
    </script>
</body>
</html>
""".replace('DATA_PLACEHOLDER', json_data)

HTML(html_content)

## Wrap Up Discussion

- What are key strengths and pitfalls that you have encountered using LLMs in this tutorial as well as looking into the future in your own research endeavors?
- Are there best practices you would employ for validating AI-generated code and maintaining reproducibility?



## Some Resources and References

- [NASA Harmonized Landsat Sentinel (HLS) Project](https://hls.gsfc.nasa.gov/)
- [US Census Bureau API](https://www.census.gov/data/developers/data-sets.html)
- [EPA's Environmental Justice Screening Tool (EJScreen)](https://www.epa.gov/ejscreen)
- [NASA SEDAC - Socioeconomic Data and Applications Center](https://sedac.ciesin.columbia.edu/)
- [LP DAAC - Getting Started with Cloud-Native HLS Data in Python](https://lpdaac.usgs.gov/resources/e-learning/getting-started-cloud-native-hls-data-python/)
- [Census Data API User Guide](https://www.census.gov/data/developers/guidance/api-user-guide.html)